In [1]:
from pathlib import Path
from PIL import Image, ImageOps
from PIL.Image import Image as PILImage
from jinja2 import Environment, FileSystemLoader

# project_root: Path = Path(r"S:\src\Richard\unknown-horizons-godot")
# path_to_images: Path = Path(r"S:\src\Richard\unknown-horizons-original\content\gfx")
project_root: Path = Path(r"S:\src\unknown-horizons-godot-port")
path_to_images: Path = Path(r"S:\src\unknown-horizons\content\gfx")

class SpriteFrameAtlas:
  atlas_path: Path
  frame_width: int
  frame_height: int
  cols: int
  rows: int
  rotations: list[int]
  animation_name: str
  def __init__(self, atlas_path: Path, frame_width: int, frame_height: int, cols: int, rows: int, rotations: list[int], animation_name: str) -> None:
    self.atlas_path = atlas_path
    self.frame_width = frame_width
    self.frame_height = frame_height
    self.cols = cols
    self.rows = rows
    self.rotations = rotations
    self.animation_name = animation_name
  def __repr__(self):
    return (f"SpriteImage(atlas_path={self.atlas_path}, frame_width={self.frame_width}, frame_height={self.frame_height}, " + 
            f"cols={self.cols}, rows={self.rows}, rotations={self.rotations}, animation_name={self.animation_name})")

def write_template(output_path: Path, template_str: str, args: dict):
  print(f"Writing template: {output_path}")
  env = Environment(loader=FileSystemLoader("."))
  template = env.from_string(template_str.strip())
  rendered_content = template.render(args)
  output_path.write_text(rendered_content)

In [17]:
sprite_frames_template: str = """
[gd_resource type="SpriteFrames" load_steps=1{#let the godot engine correct it#} format=3 uid="{{uid}}"]

{% for atlas in images -%}
[ext_resource type="Texture2D" uid="uid://{{atlas.animation_name.replace("_", "")}}atlas" path="res://{{atlas.atlas_path.relative_to(root).as_posix()}}" id="{{atlas.animation_name}}_atlas"]
{% endfor %}
{% for atlas in images %}
  {%- for row in range(atlas.rows|int)%}
    {%- for col in range(atlas.cols|int)%}

[sub_resource type="AtlasTexture" id="{{atlas.animation_name}}_{{row}}_{{col}}"]
atlas = ExtResource("{{atlas.animation_name}}_atlas")
region = Rect2({{col * atlas.frame_width}}, {{row * atlas.frame_height}}, {{atlas.frame_width}}, {{atlas.frame_height}})
    {%- endfor %}
  {%- endfor %}
{% endfor %}
[resource]
animations = [{
"frames": [{
"duration": 0.01,
"texture": null
}],
"loop": false,
"name": &"Empty",
"speed": 5.0
},
{%- for atlas in images %}
  {%- for row in range(0, atlas.rows | int)%}
{
"frames": [
    {%- for col in range(0, atlas.cols | int)%}
{
"duration": 1.0,
"texture": SubResource("{{atlas.animation_name}}_{{row}}_{{col}}")
},
    {%- endfor %}
],
"loop": true,
"name": &"{{atlas.animation_name}}_{{atlas.rotations[row%(atlas.rotations|length)]}}",{#remember that the first charecters of the atlas name are the name of bulding which can be determined from the path#}
"speed": 5.0
},
  {%- endfor %}
{% endfor %}
]
"""

In [18]:
def merge_atlas(rows: list[list[Path]]) -> PILImage:
  atlas: PILImage = Image.new("RGBA", (0, 0))
  paste_position_y: int = 0
  for row in rows:
    paste_position_x = 0
    for file in row:
      png = Image.open(file)
      expand_width: int = max(paste_position_x + png.width - atlas.width, 0)
      expand_height: int = max(paste_position_y + png.height - atlas.height, 0)
      atlas = ImageOps.expand(atlas, (0, 0, expand_width, expand_height), (0,0,0,0))
      atlas.paste(png, (paste_position_x, paste_position_y))
      paste_position_x += png.width
    paste_position_y = atlas.height
  return atlas

def generate_atlases(object_name: str, save_path: Path) -> list[SpriteFrameAtlas]:
  object_name_snake_case: str = object_name[0].lower()
  for char in object_name[1:]:
    if char.isupper():
      object_name_snake_case += "_"
    object_name_snake_case += char.lower()
  
  atlases: list[SpriteFrameAtlas] = []

  object_teirs: list[Path] = [folder for folder in path_to_images.glob("**") if folder.stem == object_name_snake_case]
  for states_for_tier_folder in object_teirs: # got through each tier-grouped folder
    object_states: list[Path] = [state for state in states_for_tier_folder.glob("*/*") if state.is_dir()] # get the rotations
    for rotations_for_state_folder in object_states: # go through the folders that represent states
      object_rotations: list[Path] = [rotation for rotation in rotations_for_state_folder.glob("*") if rotation.is_dir() and rotation.stem.isnumeric()]
      rows: list[list[Path]] = []
      for frames_for_rotation_folder in object_rotations: # go through the folders that represent the frames
        row: list[Path] = [frame for frame in frames_for_rotation_folder.glob("*") if frame.suffix == ".png"]
        rows.append(row)
      
      # get the merged atlas
      atlas_png: PILImage = merge_atlas(rows)
      # get a random image for getting the width and height
      random_image: PILImage = Image.open(rows[0][0])
      # get the tier
      tier: str = "sailors" if "units" in str(rotations_for_state_folder) else rotations_for_state_folder.parent.parent.parent.stem.lower()
      # get the state
      frame_state: str = rotations_for_state_folder.stem
      animation_name: str = f"{tier}_{frame_state}"
      # get the path at which the atlas will be saved
      atlas_path: Path = save_path / rf"{object_name_snake_case}_{animation_name}_atlas.png"
      # get the width and height of a usual image
      frame_width: int = random_image.width
      frame_height: int = random_image.height
      # get the number of columns and rows
      cols_cnt: int = int(atlas_png.size[0] / frame_width)
      rows_cnt: int = int(atlas_png.size[1] / frame_height)
      # get the list of different rotations for the atlas
      rotation_amount: int = len(rows)
      assert rotation_amount % 4 == 0
      rotations: list[int] = [rotation*45 for rotation in range(2 - int(rotation_amount/4), 8, 3 - int(rotation_amount/4))]
      # save the atlas
      atlas_png.save(atlas_path, "PNG")
      # create a sprite frame atlas for the atlas and add it to the list of sprite frame atlases
      atlas: SpriteFrameAtlas = SpriteFrameAtlas(atlas_path, frame_width, frame_height, cols_cnt, rows_cnt, rotations, animation_name)
      atlases.append(atlas)
  print(f"Atlases: {atlases}")
  return atlases

def generate_uid(building_name: str, file_id: str) -> str:
  valid_uid: str = "c"
  # add the building name to the uid
  stripped_building_name: str = building_name.lower().replace("_", "").replace("z", "").replace("9", "") # strip the building name of unallowed characters
  building_name_cutted: str = stripped_building_name[:min(len(stripped_building_name), 11)] # cut the building name to leave room for the first letter of id
  valid_uid += building_name_cutted
  # add the file id to the uid
  stripped_file_id: str = file_id.replace("_", "").lower().replace("z", "").replace("9", "") # strip the file id of unallowed characters
  file_id_cutted: str = stripped_file_id[:min(len(stripped_file_id), 13 - len(valid_uid))] # cut the file id to fit the remaining space in the uid to 13 characters
  valid_uid += file_id_cutted
  # add a spacer to make the uid 13 characters
  spacer: str = "0" * (13 - len(valid_uid))
  valid_uid += spacer
  # print and return the uid
  print(f"Generated uid: {valid_uid}")
  return valid_uid

In [21]:
object_name: str = "Warehouse"
save_path: Path = project_root / r"Assets\World\Buildings\Warehouse\Sprites"

atlases: list[SpriteFrameAtlas] = generate_atlases(object_name, save_path)
uid: str = generate_uid(save_path.parent.stem, "frames")
write_template(save_path / Path(save_path.parent.stem + "Frames.tres"), sprite_frames_template, {"images": atlases, "root": project_root, "uid": uid})

Atlases: [SpriteImage(atlas_path=S:\src\unknown-horizons-godot-port\Assets\World\Buildings\Warehouse\Sprites\warehouse_citizens_idle_atlas.png, frame_width=192, frame_height=192, cols=1, rows=4, rotations=[45, 135, 225, 315], animation_name=citizens_idle), SpriteImage(atlas_path=S:\src\unknown-horizons-godot-port\Assets\World\Buildings\Warehouse\Sprites\warehouse_pioneers_idle_atlas.png, frame_width=192, frame_height=192, cols=1, rows=4, rotations=[45, 135, 225, 315], animation_name=pioneers_idle), SpriteImage(atlas_path=S:\src\unknown-horizons-godot-port\Assets\World\Buildings\Warehouse\Sprites\warehouse_sailors_idle_atlas.png, frame_width=192, frame_height=192, cols=1, rows=3, rotations=[45, 135, 225, 315], animation_name=sailors_idle), SpriteImage(atlas_path=S:\src\unknown-horizons-godot-port\Assets\World\Buildings\Warehouse\Sprites\warehouse_settlers_idle_atlas.png, frame_width=192, frame_height=192, cols=1, rows=4, rotations=[45, 135, 225, 315], animation_name=settlers_idle)]
Gene

In [13]:
buildings_folder = Path(r"s:\src\unknown-horizons\content\gfx\buildings")
# [str(p.relative_to(buildings_folder)) for p in buildings_folder.glob("**/*") if p.is_file()]
for folder in sorted(buildings_folder.rglob("*")):
  if folder.is_dir():
    files = [f.name for f in folder.iterdir() if f.is_file()]
    print(f"{folder.relative_to(buildings_folder)}: {' '.join(files)}")

citizens: 
citizens\as_alvearies0: 
citizens\as_alvearies0\idle: tm_2000
citizens\as_alvearies0\idle\135: 000.png 001.png 002.png 003.png 004.png 005.png 006.png 007.png 008.png 009.png 010.png 011.png 012.png 013.png 014.png 015.png 016.png 017.png 018.png 019.png
citizens\as_alvearies0\idle\225: 000.png 001.png 002.png 003.png 004.png 005.png 006.png 007.png 008.png 009.png 010.png 011.png 012.png 013.png 014.png 015.png 016.png 017.png 018.png 019.png
citizens\as_alvearies0\idle\315: 000.png 001.png 002.png 003.png 004.png 005.png 006.png 007.png 008.png 009.png 010.png 011.png 012.png 013.png 014.png 015.png 016.png 017.png 018.png 019.png
citizens\as_alvearies0\idle\45: 000.png 001.png 002.png 003.png 004.png 005.png 006.png 007.png 008.png 009.png 010.png 011.png 012.png 013.png 014.png 015.png 016.png 017.png 018.png 019.png
citizens\as_cannonfoundry: 
citizens\as_cannonfoundry\idle: 
citizens\as_cannonfoundry\idle\135: 001.png
citizens\as_cannonfoundry\idle\225: 001.png
citizen